## **Practices** - *2° Partial*
* November 18°, 2025
#### ESCOM - IPN: *Natural Language Processing*
#### Prof. Marco Antonio
#### *B.S. in Data Science* - 6AV1
> Sánchez García Miguel Alexander


#### **1° Practice - Documents Classification**


**a.** Import data


> Data get from different PDF files.


In [1]:
"""Function to extract text from a PDF file starting from a specified page."""
def extract_text_from_pdf(file_path, start_page=2):
    try:
        import PyPDF2
    except ImportError:
        raise ImportError("PyPDF2 no está instalado. Instálalo con: pip install PyPDF2")
    text_pages = []
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        # Extraer texto desde start_page hasta el final
        for page_num in range(start_page, len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text_pages.append(page.extract_text() or "")
    return "\n".join(text_pages)


In [2]:
# For the training data
train_data = []
train_file_paths = [
    "Data/Deportes/Deportes_1.pdf",
    "Data/Deportes/Deportes_2.pdf",
    "Data/Derechos_Humanos/Derechos_Humanos_1.pdf",
    "Data/Derechos_Humanos/Derechos_Humanos_2.pdf",
    "Data/Politica/Politica_1.pdf",
    "Data/Politica/Politica_2.pdf",
    "Data/Videojuegos/Videojuegos_1.pdf",
    "Data/Videojuegos/Videojuegos_2.pdf"
]
# For the test data
test_data = []
test_file_paths = [
    "Data/Test/Derechos_Humanos.pdf",
    "Data/Test/Deportes.pdf",
    "Data/Test/Politica.pdf",
    "Data/Test/Videojuegos.pdf"
]


**b.** Data Preprocessing:


> We need to mention that the NLP_Utilities/ folder contains all the modules to perfomm the tasks. **(TF-IDF and BagOfWords already clean and preprocess the data)**.


In [3]:
from NLP_Utilities.TF_IDF import TF_IDF
tf_idf = TF_IDF()
tf_idf.fit_transform(docs=[text for text, label in train_data], labels=[label for text, label in train_data])
tf_idf_matrix = tf_idf.compute_tf_idf()
tf_idf_matrix


""


In [4]:
import numpy as np
class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
    def fit(self, X):
        # Center the data
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean
        # Compute covariance matrix
        cov_matrix = np.cov(X_centered, rowvar=False)
        # Eigen decomposition
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        # Sort eigenvalues and corresponding eigenvectors
        sorted_indices = np.argsort(eigenvalues)[::-1]
        sorted_eigenvalues = eigenvalues[sorted_indices]
        sorted_eigenvectors = eigenvectors[:, sorted_indices]
        # Select the top n_components
        self.components = sorted_eigenvectors[:, :self.n_components]
    def transform(self, X):
        X_centered = X - self.mean
        return np.dot(X_centered, self.components)
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


### **3° Practice - Concurrency in NLP**


**a.** Import and extract text from documents


In [5]:
# Define the paths to the documents
doc_paths = [
    "docs/constitucion.pdf",
    "docs/quijote.pdf",
    "docs/recetario.pdf"
]
# Extract text from each document
documents = []
doc_titles = []
for doc_path in doc_paths:
    # Extract document name without extension
    doc_name = doc_path.split('/')[-1].replace('.pdf', '').title()
    doc_titles.append(doc_name)
    # Extract text from PDF
    text = extract_text_from_pdf(doc_path, start_page=0)
    documents.append(text)
print(f"Loaded {len(documents)} documents:")
for i, title in enumerate(doc_titles):
    print(f"  {i+1}. {title} - {len(documents[i])} characters")


Loaded 3 documents:
  1. Constitucion - 1185218 characters
  2. Quijote - 308955 characters
  3. Recetario - 450219 characters


In [ ]:
import importlib
import NLP_Utilities.CoOccurrence_PCA
importlib.reload(NLP_Utilities.CoOccurrence_PCA)
from NLP_Utilities.CoOccurrence_PCA import CoOccurrencePCA
# Initialize the model with window size of 3
cooccurrence_pca = CoOccurrencePCA(window_size=3)
# Fit and transform the documents (preprocessing included)
cooccurrence_pca.fit_transform(
    docs=documents,
    tittles=doc_titles,
    labels=doc_titles
)
print(f"Vocabulary size: {len(cooccurrence_pca.vocabulary)} unique words")


In [ ]:
# Compute the co-occurrence matrix
cooccurrence_matrix = cooccurrence_pca.compute_cooccurrence_matrix()
print(f"Co-occurrence matrix shape: {cooccurrence_matrix.shape}")
cooccurrence_matrix.iloc[ : , : ]


NameError: name 'cooccurrence_pca' is not defined

**b.** Create Co-occurrence Matrix and apply PCA


In [ ]:
# Apply PCA to reduce to 3 dimensions
reduced_matrix = cooccurrence_pca.apply_pca(n_components=3)
print(f"Reduced matrix shape: {reduced_matrix.shape}")
print(f"\nExplained variance by each component:")
for i, var in enumerate(cooccurrence_pca.get_explained_variance(), 1):
    print(f"  PC{i}: {var:.4f} ({var*100:.2f}%)")
print(f"\nCumulative variance: {cooccurrence_pca.get_cumulative_variance()[-1]:.4f} ({cooccurrence_pca.get_cumulative_variance()[-1]*100:.2f}%)")
print(f"\nFirst 10 words in reduced space:")
reduced_matrix.head(10)


Reduced matrix shape: (18259, 3)

Explained variance by each component:
  PC1: 0.2015 (20.15%)
  PC2: 0.1464 (14.64%)
  PC3: 0.1145 (11.45%)

Cumulative variance: 0.4623 (46.23%)

First 10 words in reduced space:


,PC1,PC2,PC3
0,-1.302843,-0.496978,-0.543742
000,-1.348043,-0.570362,-0.697759
01,42.129326,61.129953,116.403261
011,-1.351493,-0.572760,-0.706713
02,59.455706,94.331095,174.322084
03,26.365119,37.713616,75.062747
04,15.612754,22.526045,44.190952
05,40.137664,51.859567,104.023177
06,46.190195,69.206594,135.042967
07,17.298003,26.692420,52.349880


In [ ]:
# Get analysis summary
summary = cooccurrence_pca.visualize_summary()
print("Analysis Summary:")
print("="*60)
for key, value in summary.items():
    print(f"{key.replace('_', ' ').title()}: {value}")


Analysis Summary:
Num Documents: 3
Vocabulary Size: 18259
Window Size: 3
Cooccurrence Matrix Shape: (18259, 18259)
Reduced Matrix Shape: (18259, 3)
N Components: 3
Explained Variance: [0.20146944668950437, 0.14635100626099495, 0.1144653297471632]
Cumulative Variance: [0.20146944668950437, 0.34782045295049935, 0.4622857826976625]


**e.** Summary statistics


In [ ]:
# Alternative visualization: words with highest variance (Interactive)
fig = cooccurrence_pca.plot_3d_interactive(
    top_n=50,
    method='variance',
    title='3D PCA Visualization of Top 50 Words by Variance'
)
fig.show()


In [ ]:
# Plot the 30 most frequent words in 3D space (Interactive)
fig = cooccurrence_pca.plot_3d_interactive(
    top_n=30,
    method='frequency',
    title='3D PCA Visualization of Top 30 Most Frequent Words'
)
fig.show()


**d.** Visualize the most relevant words in 3D


In [ ]:
# Find similar words for some example words from different documents
example_words = ['ley', 'caballero', 'cocinar', 'gobierno', 'sexo']
print("Word Similarity Analysis:\n" + "="*60)
for word in example_words:
    if word in cooccurrence_pca.vocabulary:
        print(f"\nWords most similar to '{word}':")
        similar_words = cooccurrence_pca.get_similar_words(word, top_n=5)
        print(similar_words.to_string(index=False))
    else:
        print(f"\n'{word}' not found in vocabulary")


Word Similarity Analysis:

Words most similar to 'ley':
     Word  Similarity
  recurso    0.999931
    dicha    0.999854
     caso    0.999834
   grande    0.999629
actividad    0.999622

Words most similar to 'caballero':
        Word  Similarity
asentamiento    0.999634
     pasaran    0.999546
      alcanc    0.999155
  generacion    0.999126
      basico    0.999046

Words most similar to 'cocinar':
    Word  Similarity
    1929    0.999768
promover    0.999060
       s    0.999029
   filet    0.998972
 marinar    0.998667

Words most similar to 'gobierno':
    Word  Similarity
realizar    0.999509
 federal    0.998752
   seran    0.998686
    pena    0.998263
    dema    0.998125

Words most similar to 'sexo':
        Word  Similarity
       tomen    0.999989
        coup    0.999975
    permitan    0.999968
desarrollara    0.999921
 desarrollar    0.999915
